In [1]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 46.1 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.5/29.5 MB 49.5 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 104.9 MB/s  0:00:00m0:00:0100:01
  Attempting uninstall: nvidia-cudnn-cu12━━━━━━━ 0/5 [opencv-python]
    Found existing installation: nvidia-cudnn-cu12 9.3.0.750m [opencv-python]
    Uninstalling nvidia-cudnn-cu12-9.3.0.75: 0/5 [opencv-python]
      Successfully uninstalled nvidia-cudnn-cu12-9.3.0.75━━━━━━━━━ 1/5 [nvidia-cudnn-cu12]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [configargparse]m [imageio]ffmpeg]12]


In [9]:
!pip install scikit-image

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 119.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-image] [scikit-image]


In [3]:
!python src/run_platonerf.py --config configs/chair.txt

/opt/conda/lib/python3.10/site-packages/torch/__init__.py:1144: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:432.)
  _C._set_default_tensor_type(t)
Not loading ToF image 0.
Not loading ToF image 1.
Not loading ToF image 2.
Not loading ToF image 3.
Not loading ToF image 10.
Not loading ToF image 13.
Not loading ToF image 20.
Not loading ToF image 22.
Loaded ToF data (16, 512, 512, 391) (16, 3) (16, 3) [512, 512, np.float64(256.00000000000006)] ./data/chair (16, 11) (16, 11)
Train idxs: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]
Found ckpts []
Not ndc!
get rays
The following projected lights have been found: [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11], [12], [13], [14], [15]]
Preprocessing tof image 1 of 16.
[ WARN:0@88.257] global loadsave.cpp:1089 imwrite_ Unsupported depth image fo

In [8]:
!python src/render_test_depth.py --config configs/bunny.txt --ft_path pretrained/bunny.tar --output_dir logs/bunny --raw_noise_std 3.0


/opt/conda/lib/python3.10/site-packages/torch/__init__.py:1144: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:432.)
  _C._set_default_tensor_type(t)
Loaded ToF data (24, 512, 512, 391) (24, 3) (24, 3) [512, 512, np.float64(256.00000000000006)] ./data/bunny (24, 11) (24, 11)
Train idxs: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]
Found ckpts ['pretrained/bunny.tar']
Reloading from pretrained/bunny.tar
/root/myhomedir/bucket/CI/PROJET/PlatoNeRF/src/render_test_depth.py:179: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#un

In [1]:
import numpy as np
import os
import math
import glob

def calculate_psnr(img1, img2, max_val):
    """Calcule le PSNR entre deux matrices numpy."""
    mse = np.mean((img1 - img2) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * math.log10((max_val ** 2) / mse)

def evaluate_folders(dir_ref, dir_exp):
    # On récupère la liste des fichiers .npy triés
    files_ref = sorted(glob.glob(os.path.join(dir_ref, "*.npy")))
    files_exp = sorted(glob.glob(os.path.join(dir_exp, "*.npy")))
    
    if len(files_ref) != len(files_exp) or len(files_ref) == 0:
        print("Erreur : Les dossiers ne contiennent pas le même nombre de fichiers .npy ou sont vides.")
        return

    psnr_list = []
    
    MAX_DEPTH = 6.0 

    for f_ref, f_exp in zip(files_ref, files_exp):
        depth_ref = np.load(f_ref)
        depth_exp = np.load(f_exp)
        #PSNR
        psnr_val = calculate_psnr(depth_ref, depth_exp, max_val=MAX_DEPTH)
        psnr_list.append(psnr_val)
        
        print(f"{os.path.basename(f_ref)} -> PSNR: {psnr_val:.2f} dB")
        
    avg_psnr = np.mean(psnr_list)
    print("-" * 30)
    print(f"PSNR Moyen sur la séquence : {avg_psnr:.2f} dB")

if __name__ == '__main__':
    dossier_reference = "logs/chair/depth_predictions_n_samples1024"
    dossier_experience = "logs/chair/depth_predictions_n_samples16"
    
    evaluate_folders(dossier_reference, dossier_experience)

depth_map_000.npy -> PSNR: 13.36 dB
depth_map_001.npy -> PSNR: 18.87 dB
depth_map_002.npy -> PSNR: 21.62 dB
depth_map_003.npy -> PSNR: 22.71 dB
depth_map_004.npy -> PSNR: 24.59 dB
depth_map_005.npy -> PSNR: 17.69 dB
depth_map_006.npy -> PSNR: 24.09 dB
------------------------------
PSNR Moyen sur la séquence : 20.42 dB


In [2]:

from skimage.metrics import structural_similarity as ssim

def calculate_psnr(img1, img2, max_val):
    """Calcule le PSNR entre deux matrices numpy (cartes de profondeur)."""
    mse = np.mean((img1 - img2) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * math.log10((max_val ** 2) / mse)

def evaluate_folders(dir_ref, dir_exp, max_depth=6.0):
    files_ref = sorted(glob.glob(os.path.join(dir_ref, "*.npy")))
    files_exp = sorted(glob.glob(os.path.join(dir_exp, "*.npy")))
    
    if len(files_ref) != len(files_exp) or len(files_ref) == 0:
        print("Erreur : Les dossiers ne contiennent pas le même nombre de fichiers .npy ou sont vides.")
        return

    psnr_list = []
    ssim_list = []

    for f_ref, f_exp in zip(files_ref, files_exp):
        depth_ref = np.load(f_ref)
        depth_exp = np.load(f_exp)
        
        # PSNR
        psnr_val = calculate_psnr(depth_ref, depth_exp, max_val=max_depth)
        psnr_list.append(psnr_val)
        
        # SSIM
        ssim_val = ssim(depth_ref, depth_exp, data_range=max_depth)
        ssim_list.append(ssim_val)
        
        filename = os.path.basename(f_ref)
        print(f"{filename} -> PSNR: {psnr_val:.2f} dB | SSIM: {ssim_val:.4f}")

    print("-" * 40)
    print(f"Comparaison terminée sur {len(files_ref)} images.")
    print(f"PSNR Moyen : {np.mean(psnr_list):.2f} dB")
    print(f"SSIM Moyen : {np.mean(ssim_list):.4f} (proche de 1.0 = parfait)")

if __name__ == '__main__':
    dossier_reference = "logs/chair/depth_predictions_n_samples256"
    dossier_experience = "logs/chair/depth_predictions_n_samples64"
    
    evaluate_folders(dossier_reference, dossier_experience, max_depth=6.0)

ModuleNotFoundError: No module named 'skimage'

In [ ]:
PSNR 27.83 et SSIM 0.7034 pour bunny 
PSNR 28.05 et SSIM Moyen 0.7540 pour chair